In [2]:
from langchain_community.llms import Ollama
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.docstore.document import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain.chains import RetrievalQA

c:\Users\ikath\OneDrive\GitHub\TrustworthyRAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# 1. Load documents (example: PDF)
with open("../../data/raw/pride_and_prejudice.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_text(raw_text)
print(f"Number of chunks: {len(chunks)}")

documents = [Document(page_content=chunk) for chunk in chunks]
print(f"First chunk content: {documents[0].page_content[:500]}...")

Number of chunks: 1293
First chunk content: The Project Gutenberg eBook of Pride and Prejudice
    
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this ebook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

...


In [7]:
# 2. Create embeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")

C:\Users\ikath\AppData\Local\Temp\ipykernel_21496\1890460720.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [8]:
# 3. Store vectors
vectordb = Chroma.from_documents(documents, embedding=embeddings, persist_directory="./chroma_db")

KeyboardInterrupt: 

In [ ]:
# 4. Create retriever
retriever = vectordb.as_retriever()

In [ ]:
# 5. LLM with Ollama
llm = Ollama(model="mistral")

In [ ]:
# 6. Build RAG pipeline
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)

In [ ]:
# 7. Ask questions

while True:
	query = input("\nEnter your question (or 'exit' to quit): ")
	if query.lower() == 'exit':
		break

	answer = qa_chain.run(query)
	print("\nAnswer:", answer)